In [ ]:
import copy
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader


# 1. Check that the shared model and evaluation functions are available
required_objects = [
    "GCNForecaster",
    "calculate_metrics",
    "evaluate_model"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run Code 7 first. Missing objects: "
        + ", ".join(missing_objects)
    )


# 2. Define paths and training settings
graph_folder = Path(
    "results/step_06"
)

baseline_folder = Path(
    "results/step_07"
)

output_folder = Path(
    "results/step_12"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

SEED = 42
HIDDEN_CHANNELS = 16
LEARNING_RATE = 0.001
EPOCHS = 200
BATCH_SIZE = 16

MODEL_NAME = "GCN_without_score"

device = torch.device("cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_num_threads(2)

torch.use_deterministic_algorithms(
    True,
    warn_only=True
)


# 3. Verify agreement with the original GCN training settings
baseline_configuration = json.loads(
    (
        baseline_folder / "GCN_configuration.json"
    ).read_text(encoding="utf-8")
)

assert baseline_configuration["seed"] == SEED
assert baseline_configuration["epochs"] == EPOCHS

assert (
    baseline_configuration["hidden_channels"]
    == HIDDEN_CHANNELS
)

assert np.isclose(
    baseline_configuration["learning_rate"],
    LEARNING_RATE
)

assert (
    baseline_configuration["batch_size"]
    == BATCH_SIZE
)


# 4. Load the original graph datasets
payload = torch.load(
    graph_folder / "graph_datasets.pt",
    map_location="cpu",
    weights_only=True
)

references = payload["references"]
feature_names = payload["feature_names"]

target_scale = float(payload["target_scale"])
n_parts = int(payload["n_parts"])
n_features = int(payload["n_features"])
forecast_horizon = int(payload["forecast_horizon"])

score_index = feature_names.index(
    "Fuzzy_TOPSIS_Score"
)


# 5. Neutralize the score in every graph
def remove_score_information(graph_dictionaries):

    modified_graphs = []

    for graph_dictionary in graph_dictionaries:

        original_graph = Data(
            **graph_dictionary
        )

        modified_graph = original_graph.clone()

        # Zero is a constant in the standardized feature space.
        # Retraining prevents the model from relying on the score.
        modified_graph.x[:, score_index] = 0.0

        # Verify that all other features remain unchanged.
        other_indices = [
            index
            for index in range(n_features)
            if index != score_index
        ]

        assert torch.equal(
            modified_graph.x[:, other_indices],
            original_graph.x[:, other_indices]
        )

        assert torch.equal(
            modified_graph.edge_index,
            original_graph.edge_index
        )

        assert torch.equal(
            modified_graph.y,
            original_graph.y
        )

        assert torch.count_nonzero(
            modified_graph.x[:, score_index]
        ).item() == 0

        modified_graphs.append(
            modified_graph
        )

    return modified_graphs


train_graphs = remove_score_information(
    payload["train"]
)

validation_graphs = remove_score_information(
    payload["validation"]
)

test_graphs = remove_score_information(
    payload["test"]
)


# 6. Create the data loaders using the original sampling settings
train_loader = DataLoader(
    train_graphs,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED)
)

validation_loader = DataLoader(
    validation_graphs,
    batch_size=1,
    shuffle=False
)

test_loader = DataLoader(
    test_graphs,
    batch_size=1,
    shuffle=False
)


# 7. Initialize a new GCN rather than reusing trained weights
model = GCNForecaster(
    input_channels=n_features,
    hidden_channels=HIDDEN_CHANNELS,
    prediction_horizon=forecast_horizon
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

loss_function = nn.MSELoss()


# 8. Retrain the GCN without score information
history = []

best_validation_loss = float("inf")
best_state = None
best_epoch = None

for epoch in range(1, EPOCHS + 1):

    model.train()

    training_loss_sum = 0.0
    training_value_count = 0

    for graph in train_loader:

        graph = graph.to(device)

        optimizer.zero_grad()

        predictions = model(graph)

        loss = loss_function(
            predictions,
            graph.y
        )

        loss.backward()
        optimizer.step()

        training_loss_sum += (
            loss.item() * graph.y.numel()
        )

        training_value_count += (
            graph.y.numel()
        )

    training_loss = (
        training_loss_sum
        / training_value_count
    )

    validation_loss, validation_metrics = evaluate_model(
        model,
        validation_loader
    )

    history.append({
        "Epoch": epoch,
        "Training_MSE_Scaled": training_loss,
        "Validation_MSE_Scaled": validation_loss,
        "Validation_MAE": validation_metrics["MAE"],
        "Validation_RMSE": validation_metrics["RMSE"]
    })

    if validation_loss < best_validation_loss:

        best_validation_loss = validation_loss

        best_state = copy.deepcopy(
            model.state_dict()
        )

        best_epoch = epoch

    if epoch == 1 or epoch % 25 == 0:

        print(
            f"Epoch {epoch:03d}/{EPOCHS} | "
            f"Training loss: {training_loss:.5f} | "
            f"Validation loss: {validation_loss:.5f} | "
            f"Validation MAE: "
            f"{validation_metrics['MAE']:.3f}"
        )


# 9. Restore and save the best ablation model
model.load_state_dict(
    best_state
)

model.eval()

torch.save(
    {
        "state_dict": best_state,
        "input_channels": n_features,
        "hidden_channels": HIDDEN_CHANNELS,
        "prediction_horizon": forecast_horizon,
        "target_scale": target_scale,
        "references": references,
        "feature_names": feature_names,
        "neutralized_feature": "Fuzzy_TOPSIS_Score",
        "neutralized_feature_index": score_index,
        "best_epoch": best_epoch,
        "best_validation_loss": best_validation_loss,
        "seed": SEED
    },
    output_folder / "GCN_without_score_best_model.pt"
)

history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    output_folder / "GCN_without_score_training_history.csv",
    index=False
)


# 10. Retrieve the forecast origin dates
window_table = pd.read_csv(
    "results/step_05/forecast_windows.csv",
    parse_dates=["Origin_Date"]
)

origin_dates = (
    window_table
    .set_index("Origin_Index")["Origin_Date"]
    .to_dict()
)


# 11. Generate predictions using the ablation model
prediction_records = []

with torch.no_grad():

    for graph in test_loader:

        graph = graph.to(device)

        predictions = model(graph)

        predictions_raw = np.maximum(
            predictions.cpu().numpy() * target_scale,
            0
        )

        actual_raw = (
            graph.y_raw.cpu().numpy()
        )

        origin_index = int(
            graph.origin.item()
        )

        origin_date = pd.Timestamp(
            origin_dates[origin_index]
        )

        for part_index, reference in enumerate(references):

            for horizon in range(1, forecast_horizon + 1):

                target_date = (
                    origin_date
                    + pd.DateOffset(months=horizon)
                )

                prediction_records.append({
                    "Model": MODEL_NAME,
                    "Reference": reference,
                    "Origin_Date": origin_date.strftime(
                        "%Y-%m-%d"
                    ),
                    "Target_Date": target_date.strftime(
                        "%Y-%m-%d"
                    ),
                    "Horizon": horizon,
                    "Actual": float(
                        actual_raw[
                            part_index,
                            horizon - 1
                        ]
                    ),
                    "Prediction": float(
                        predictions_raw[
                            part_index,
                            horizon - 1
                        ]
                    ),
                    "Is_Synthetic": True
                })

ablation_predictions = pd.DataFrame(
    prediction_records
)

ablation_predictions.to_csv(
    output_folder / "GCN_without_score_test_predictions.csv",
    index=False
)


# 12. Load the original GCN predictions
baseline_predictions = pd.read_csv(
    baseline_folder / "GCN_test_predictions.csv"
)

expected_targets = pd.read_csv(
    "results/step_05/test_target_identities.csv"
)

identity_columns = [
    "Reference",
    "Origin_Date",
    "Target_Date",
    "Horizon",
    "Actual"
]

expected_identities = (
    expected_targets[identity_columns]
    .sort_values(["Reference", "Horizon"])
    .reset_index(drop=True)
)

# Verify both models against the common test protocol.
for prediction_table in [
    baseline_predictions,
    ablation_predictions
]:

    identities = (
        prediction_table[identity_columns]
        .sort_values(["Reference", "Horizon"])
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        identities,
        expected_identities,
        check_dtype=False,
        check_exact=False,
        rtol=1e-6,
        atol=1e-6
    )


# 13. Recalculate the metrics for both models
baseline_metrics = calculate_metrics(
    baseline_predictions["Actual"],
    baseline_predictions["Prediction"]
)

ablation_metrics = calculate_metrics(
    ablation_predictions["Actual"],
    ablation_predictions["Prediction"]
)

comparison_df = pd.DataFrame([
    {
        "Model": "GCN_with_score",
        **baseline_metrics
    },
    {
        "Model": "GCN_without_score",
        **ablation_metrics
    }
])

comparison_df.to_csv(
    output_folder / "GCN_ablation_comparison.csv",
    index=False
)

pd.DataFrame([
    {
        "Model": MODEL_NAME,
        **ablation_metrics
    }
]).to_csv(
    output_folder / "GCN_without_score_test_metrics.csv",
    index=False
)


# 14. Quantify the observed benefit of including the score
# Positive values mean the model WITH the score has lower error.
effect_records = []

for metric in [
    "MAE",
    "RMSE",
    "MAPE_nonzero_pct",
    "WAPE_pct"
]:

    with_score = baseline_metrics[metric]
    without_score = ablation_metrics[metric]

    if with_score is None or without_score is None:

        difference = None
        relative_gain = None

    else:

        difference = (
            without_score - with_score
        )

        relative_gain = (
            100 * difference / without_score
            if without_score > 0
            else None
        )

    effect_records.append({
        "Metric": metric,
        "With_Score": with_score,
        "Without_Score": without_score,
        "Error_Without_Minus_With": difference,
        "Relative_Error_Reduction_Pct": relative_gain
    })

effect_df = pd.DataFrame(
    effect_records
)

effect_df.to_csv(
    output_folder / "GCN_score_ablation_effect.csv",
    index=False
)

print("\nGCN ablation comparison:")
print(
    comparison_df.round(4).to_string(index=False)
)

print("\nObserved effect of including the score:")
print(
    effect_df.round(4).to_string(index=False)
)


# 15. Compare the models separately at each horizon
horizon_records = []

for name, prediction_table in [
    ("GCN_with_score", baseline_predictions),
    ("GCN_without_score", ablation_predictions)
]:

    for horizon, group in prediction_table.groupby("Horizon"):

        horizon_records.append({
            "Model": name,
            "Horizon": int(horizon),
            **calculate_metrics(
                group["Actual"],
                group["Prediction"]
            )
        })

pd.DataFrame(
    horizon_records
).to_csv(
    output_folder / "GCN_ablation_by_horizon.csv",
    index=False
)


# 16. Plot the ablation comparison
fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 4)
)

labels = [
    "With score",
    "Without score"
]

for ax, metric in zip(
    axes,
    ["MAE", "RMSE"]
):

    ax.bar(
        labels,
        comparison_df[metric],
        color=["#4472C4", "#ED7D31"]
    )

    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.grid(axis="y", alpha=0.25)

fig.suptitle(
    "GCN score ablation "
)

fig.tight_layout()

fig.savefig(
    output_folder / "GCN_ablation_comparison.png",
    dpi=160
)

plt.show()
plt.close(fig)


# 17. Save the ablation protocol
configuration = {
    "model": MODEL_NAME,
    "neutralized_feature": "Fuzzy_TOPSIS_Score",
    "neutralization": (
        "Set the standardized score column to zero "
        "during training, validation, and testing."
    ),
    "trained_from_scratch": True,
    "same_architecture": True,
    "same_graph_topology": True,
    "same_other_features": True,
    "same_test_targets": True,
    "hidden_channels": HIDDEN_CHANNELS,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "best_epoch": best_epoch,
    "best_validation_loss": best_validation_loss,
    "test_metrics": ablation_metrics,
    "interpretation": (
        "This is a single-seed comparison on synthetic data. "
        "A positive error reduction favors score inclusion. "
        "No improvement or statistical significance is assumed."
    )
}

(
    output_folder / "GCN_ablation_configuration.json"
).write_text(
    json.dumps(configuration, indent=2),
    encoding="utf-8"
)

print(f"\nResults saved in: {output_folder}")